<a href="https://colab.research.google.com/github/PDavid413/PL_1142/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [35]:
!pip install -q google-generativeai

In [29]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [36]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [37]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1s2wYbhb8dq15G2rjrwT3a32UooZhc--n4T5pw6Uwc88/edit?usp=sharing"
WORKSHEET_NAME = "成績計算表"
REQUIRED_COLUMNS = ["日期", "科目", "成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [38]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="whats 9 + 10 meme"
)
print(response.text)

It's a classic Vine meme!

The "9 + 10 meme" comes from a short video where an older person asks a young kid:

**"What's 9 + 10?"**

And the kid, with absolute confidence, replies:

**"21."**

When the older person corrects him (9 + 10 is actually 19), the kid doubles down and says something along the lines of:

**"No, it's 21! You stoopid!"** (sometimes spelled "you stupid")

The humor comes from:

1.  **The sheer wrongness of the answer:** It's a simple math problem, and the kid gets it spectacularly wrong.
2.  **The kid's unwavering confidence:** Despite being incorrect, he's absolutely certain of his answer.
3.  **The immediate, aggressive, and slightly adorable defensiveness:** Instead of reconsidering, he flips the script and calls the questioner "stoopid."

It became a widely used meme to express or highlight:

*   Someone being confidently wrong.
*   A nonsensical or illogical statement.
*   A moment of playful exasperation at someone's error.
*   Just a general reference to in

### 定義 AI 摘要函式

In [44]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，分析學生的學業優勢。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(
            model = MODEL_ID,
            contents = prompt_text
        )
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{type(e).__name__} - {e}") # 印出更詳細的錯誤資訊
        return "AI 摘要生成失敗。"

In [47]:
def process_grades_and_summary(grade_data, clear_sheet_before_writing):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績] 的列表的列表，例如：[['國文', 90], ['英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade_str in grade_data:
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    sheet_message = ""
    try:
        if clear_sheet_before_writing: # New logic for 'wash out and rewrite'
            print("--- 清除 Google Sheet 內容並重新寫入標題... ---")
            _ws.clear() # Clear all content
            _ws.update('A1', [REQUIRED_COLUMNS]) # Write headers back
            last_data_row = 1 # Headers are in row 1, so data starts from row 2
            sheet_message += "Google Sheet 已清除並寫入標題。\n"
        else:
            # 找出目前工作表中最後一個有資料的列
            all_sheet_data = _ws.get_all_values()
            last_data_row = 0
            for r_idx, row in reversed(list(enumerate(all_sheet_data))): # 從底部往上找
                if any(cell.strip() for cell in row): # 檢查是否有任何儲存格有內容
                    last_data_row = r_idx + 1 # 取得 1-based 的行號
                    break

        # 新成績將從 last_data_row + 1 開始寫入
        start_row_for_grades = last_data_row + 1
        num_grade_rows = len(new_grades_for_sheet)

        if num_grade_rows > 0:
            # 計算寫入成績的範圍 (假設是A到C列)
            end_row_for_grades = start_row_for_grades + num_grade_rows - 1
            grade_range = f'A{start_row_for_grades}:C{end_row_for_grades}'
            _ws.update(grade_range, new_grades_for_sheet)
            sheet_message += "成績已成功寫入 Google Sheet。\n"
        else:
            sheet_message += "沒有新的成績需要寫入。\n"

    except Exception as e:
        sheet_message += f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    # 獲取 AI 摘要
    summary = get_ai_summary(new_grades_for_sheet)

    try:
        # AI 摘要將在成績寫入之後，緊接著開始寫入
        # 如果沒有成績被寫入，則從 last_data_row + 1 開始
        # 否則，從 end_row_for_grades + 1 開始
        start_row_for_summary = end_row_for_grades + 1 if 'end_row_for_grades' in locals() and end_row_for_grades > 0 else last_data_row + 1

        # 準備要寫入 Google Sheet 的 AI 摘要數據
        # 第一行是日期和 'AI 摘要'
        summary_data_to_write = [[datetime.now().strftime('%Y-%m-%d'), 'AI 摘要']]
        # 將摘要內容分成多行，並過濾掉空白行，然後作為第三列追加
        summary_lines = [line for line in summary.split('\n') if line.strip()]

        if summary_lines:
            # 第一行的第三列是摘要的第一部分
            summary_data_to_write[0].append(summary_lines[0])
            # 其餘的行只有第三列是摘要的內容
            for line in summary_lines[1:]:
                summary_data_to_write.append(['', '', line])
        else:
            summary_data_to_write[0].append('') # 如果沒有摘要內容，第三列留空

        # 計算批次更新的範圍
        num_summary_rows = len(summary_data_to_write)
        end_row_for_summary = start_row_for_summary + num_summary_rows - 1
        summary_range = f'A{start_row_for_summary}:C{end_row_for_summary}'

        # 執行批次更新
        _ws.update(summary_range, summary_data_to_write)
        sheet_message += "AI 摘要已成功寫入 Google Sheet。"
    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

In [49]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
today = datetime.now().strftime('%Y-%m-%d') # Define today for test data
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"],
    ["自然", "78"],
    ["社會", "92"],
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data, clear_sheet_before_writing=True)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")


--- 正在執行 process_grades_and_summary 函式單元測試... ---
--- 清除 Google Sheet 內容並重新寫入標題... ---
寫入 Google Sheet 失敗：APIError: [429]: Quota exceeded for quota metric 'Write requests' and limit 'Write requests per minute' of service 'sheets.googleapis.com' for consumer 'project_number:522309567947'.

--- 正在呼叫 AI 模型生成摘要... ---
呼叫 AI 時發生錯誤：ClientError - 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 54.399258297s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev

### 驗證 Google Sheet 工作表名稱

執行以下程式碼，它會嘗試連線到您的 Google Sheet URL 並列出其中所有工作表的名稱，這可以幫助您確認 `WORKSHEET_NAME` 是否設定正確。

In [15]:
print("--- 正在重新驗證 Google Sheet 連線狀態... ---")
auth.authenticate_user()
creds, _ = default()
_gc = gspread.authorize(creds)

try:
    sh = _gc.open_by_url(SHEET_URL)
    print(f"成功連線到 Google Sheet: {sh.title}")
    worksheets = sh.worksheets()
    print("此 Google Sheet 中所有工作表名稱：")
    for ws in worksheets:
        print(f"- {ws.title}")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"錯誤：找不到指定 URL 的 Google Sheet。請檢查 `SHEET_URL`: {SHEET_URL}")
except gspread.exceptions.NoValidUrlKeyFound:
    print(f"錯誤：`SHEET_URL` 格式無效。請確認 URL 是否正確: {SHEET_URL}")
except gspread.exceptions.WorksheetNotFound as e:
    print(f"錯誤：找不到指定名稱的工作表。請檢查 `WORKSHEET_NAME` 是否正確，錯誤訊息: {e}")
except Exception as e:
    print(f"連線到 Google Sheet 時發生未預期的錯誤: {e}")


--- 正在重新驗證 Google Sheet 連線狀態... ---
成功連線到 Google Sheet: 成績計算表
此 Google Sheet 中所有工作表名稱：
- 成績計算表


定義 Gradio 處理函式

In [20]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請在下方的表格中輸入學生的科目和成績，然後點擊『送出』。系統會將資料寫入 Google Sheet 並生成 AI 摘要。")

    with gr.Row():
        with gr.Column():
            grade_input = gr.Dataframe(
                headers=["科目", "成績"],
                value=[["", ""]],
                type="array",
                row_count=1,
                col_count=(2, "fixed"),
                label="輸入科目與成績 (點擊最後一行可新增)"
            )
            clear_sheet_checkbox = gr.Checkbox(label="寫入前清除 Google Sheet", value=False)
            submit_button = gr.Button("送出")

        with gr.Column():
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(label="AI 摘要", lines=15)

    submit_button.click(
        process_grades_and_summary,
        inputs=[grade_input, clear_sheet_checkbox],
        outputs=[sheet_output, summary_output]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d1300a77137daa6a2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
